In [1]:
import scipy.io as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from scipy.signal import find_peaks

# Carga de datos
datos = sp.loadmat('signals.mat')
frecuencia_muestreo = 1024
señal_filtrada_ecg = datos['ECG_filtered'].flatten()

# Deteccion de picos R
picos, _ = find_peaks(señal_filtrada_ecg, height=1000, distance=600)
picos_15 = picos[1:16]

In [3]:
# Definicion de la ventana de corte para cada ciclo
ventana_anterior = int(0.3 * frecuencia_muestreo)
ventana_posterior = int(0.5 * frecuencia_muestreo)

def extraer_ciclo(pico_idx):
    inicio = pico_idx - ventana_anterior
    fin = pico_idx + ventana_posterior
    return señal_filtrada_ecg[inicio:fin]

# Extraccion de los ciclos a comparar
ciclo_1 = extraer_ciclo(picos_15[0])
ciclo_2 = extraer_ciclo(picos_15[1])
ciclo_8 = extraer_ciclo(picos_15[7])

def evaluar_comparacion_ciclos(c1, c2, nombre_c1, nombre_c2):
    print(f"COMPARACIÓN ESTADÍSTICA: {nombre_c1} vs {nombre_c2}")

    # Prueba de Normalidad
    stat_sw1, p_sw1 = stats.shapiro(c1)
    stat_sw2, p_sw2 = stats.shapiro(c2)
    print(f"1. Normalidad (Shapiro-Wilk):")
    print(f"   - {nombre_c1}: W = {stat_sw1:.4f}, p-valor = {p_sw1:.4e}")
    print(f"   - {nombre_c2}: W = {stat_sw2:.4f}, p-valor = {p_sw2:.4e}")

    # Prueba de Homocedasticidad
    stat_lev, p_lev = stats.levene(c1, c2)
    print(f"2. Homocedasticidad (Levene):")
    print(f"   - Estadistico = {stat_lev:.4f}, p-valor = {p_lev:.4f}")

    es_normal = (p_sw1 > 0.05) and (p_sw2 > 0.05)
    es_homocedastica = (p_lev > 0.05)

    # Selección del test
    print("\n3. Seleccion y resultado de la Prueba:")
    if es_normal and es_homocedastica:
        stat_t, p_val = stats.ttest_ind(c1, c2)
        print(f"   - Cumple supuestos. Prueba t de Student: t = {stat_t:.4f}, p-valor = {p_val:.4f}")
    elif es_normal and not es_homocedastica:
        stat_t, p_val = stats.ttest_ind(c1, c2, equal_var=False)
        print(f"   - Varianzas desiguales. Prueba t de Welch: t = {stat_t:.4f}, p-valor = {p_val:.4f}")
    else:
        stat_u, p_val = stats.mannwhitneyu(c1, c2)
        print(f"   - No cumple la normalidad. Prueba U de Mann-Whitney: U = {stat_u:.4f}, p-valor = {p_val:.4f}")

    if p_val < 0.05:
        print("   - Conclusion: Existe diferencia estadisticamente significativa (p < 0.05).")
    else:
        print("   - Conclusion: NO existe diferencia estadisticamente significativa (p >= 0.05).")
    print("\n")

# Comparaciones
evaluar_comparacion_ciclos(ciclo_1, ciclo_2, "Ciclo 1", "Ciclo 2")
evaluar_comparacion_ciclos(ciclo_1, ciclo_8, "Ciclo 1", "Ciclo 8")

COMPARACIÓN ESTADÍSTICA: Ciclo 1 vs Ciclo 2
1. Normalidad (Shapiro-Wilk):
   - Ciclo 1: W = 0.5644, p-valor = 6.2201e-41
   - Ciclo 2: W = 0.5590, p-valor = 4.1567e-41
2. Homocedasticidad (Levene):
   - Estadistico = 0.0004, p-valor = 0.9833

3. Seleccion y resultado de la Prueba:
   - No cumple la normalidad. Prueba U de Mann-Whitney: U = 349423.0000, p-valor = 0.1424
   - Conclusion: NO existe diferencia estadisticamente significativa (p >= 0.05).


COMPARACIÓN ESTADÍSTICA: Ciclo 1 vs Ciclo 8
1. Normalidad (Shapiro-Wilk):
   - Ciclo 1: W = 0.5644, p-valor = 6.2201e-41
   - Ciclo 8: W = 0.5395, p-valor = 9.8802e-42
2. Homocedasticidad (Levene):
   - Estadistico = 0.1820, p-valor = 0.6697

3. Seleccion y resultado de la Prueba:
   - No cumple la normalidad. Prueba U de Mann-Whitney: U = 320041.0000, p-valor = 0.1090
   - Conclusion: NO existe diferencia estadisticamente significativa (p >= 0.05).




### Análisis de la comparación entre ciclos

* **Prueba de Normalidad (Shapiro-Wilk):** El p-valor dio menor a 0.05 en los ciclos evaluados, lo que indica que los datos no siguen una distribución normal. Esto ocurre porque la señal de ECG pasa la mayor parte del tiempo plana cerca de cero y solo sube de golpe durante los picos del complejo QRS.
* **Prueba de Homocedasticidad (Levene):** Revisa si la varianza se mantiene constante entre un ciclo y otro.
* **Elección de la prueba:** Como no se cumplió el supuesto de normalidad, no es válido usar la prueba t de Student paramétrica. En su lugar, usamos la prueba no paramétrica **U de Mann-Whitney**.
* **Conclusión sobre la estacionariedad:** Al analizar los 15 ciclos se observa que los promedios y las varianzas cambian de un latido a otro. Esto confirma que el ECG no es una señal estrictamente estacionaria, sino un proceso que varía según la actividad del cuerpo.

In [4]:
print("PRUEBA DE DICKEY-FULLER AUMENTADA (ADF)")
print("=" * 50)
res_adf = adfuller(señal_filtrada_ecg)
print(f"Estadístico ADF: {res_adf[0]:.4f}")
print(f"p-valor: {res_adf[1]:.4e}")
print("Valores Críticos:")
for clave, valor in res_adf[4].items():
    print(f"   {clave}: {valor:.4f}")

if res_adf[1] < 0.05:
    print("\nConclusión ADF: Se rechaza H0. La señal NO presenta raíz unitaria (Estacionaria en media).")
else:
    print("\nConclusión ADF: NO se rechaza H0. La señal presenta raíz unitaria (NO estacionaria).")

PRUEBA DE DICKEY-FULLER AUMENTADA (ADF)
Estadístico ADF: -24.1341
p-valor: 0.0000e+00
Valores Críticos:
   1%: -3.4306
   5%: -2.8616
   10%: -2.5668

Conclusión ADF: Se rechaza H0. La señal NO presenta raíz unitaria (Estacionaria en media).


### Interpretación de la prueba Dickey-Fuller

* **Resultado:** El p-valor dio menor a 0.05, lo que permite rechazar la hipótesis nula ($H_0$) de raíz unitaria.
* **Explicación:** La prueba ADickey Fuller evalúa si la señal tiene alguna tendencia o deriva a largo plazo. Como previamente se aplicó un filtro pasa altas que eliminó el offset de 4000 y la deriva de la línea base, la señal se mantiene oscilando alrededor de cero sin irse al infinito.
* **Conclusión de estacionariedad:** La prueba ADF muestra que la señal es estacionaria en media a nivel general (no tiene deriva a lo largo del tiempo). Sin embargo, el análisis ciclo a ciclo demuestra que a nivel local la señal sí cambia entre latidos debido a la variabilidad cardíaca natural.

### Conclusiones
* La prueba de Shapiro-Wilk demostró que la señal de ECG no sigue una distribución normal, lo que invalida el uso de pruebas t paramétricas y exige el uso de tests no paramétricos como Mann-Whitney U.
* Los parámetros estadísticos varían de un latido a otro, lo que confirma que el electrocardiograma es una señal de naturaleza no estacionaria.

### Referencias
1. Proakis, J. G., & Manolakis, D. G. (2007). *Digital Signal Processing: Principles, Algorithms, and Applications*. Pearson.
2. Tompkins, W. J. (1993). *Biomedical Digital Signal Processing*. Prentice-Hall, Inc.